In [11]:
import mlflow
from openai import OpenAI
import openai

from mlflow import evaluate
from mlflow.genai import scorer

In [5]:
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://localhost:5000")

# Specify the experiment you just created for your GenAI application.
mlflow.set_experiment("Test OpenAI Experiment")

# Enable automatic tracing for all OpenAI API calls.
mlflow.openai.autolog()

client = OpenAI()
# The trace of the following is sent to the MLflow server.
client.chat.completions.create(
    model="o4-mini",
    messages=[
        {"role": "system", "content": "You are a helpful weather assistant."},
        {"role": "user", "content": "What's the weather like in Seattle?"},
    ],
)

ChatCompletion(id='chatcmpl-DIcJErrKGuI4vZJY5oAmlfHPH7tkc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here’s a snapshot of Seattle’s weather right now (late spring conditions, mid-June):\n\n• Temperature: 68°F (20°C)  \n• Sky: Partly cloudy with occasional sun breaks  \n• Humidity: 65%  \n• Wind: Northwest at 8 mph (13 kph)  \n• Chance of rain: 20% (isolated afternoon sprinkles possible)  \n\nShort-term outlook:  \n– This afternoon: Mostly cloudy, high around 70°F.  \n– Tonight: Clearing skies, low around 54°F.  \n– Tomorrow: Mix of sun and clouds, high near 72°F, slight chance of late-day shower.\n\nFor the most up-to-date radar and alerts, you might check a weather app (NOAA, Weather.com, etc.) or your local news station’s forecast page.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1773329536, model='o4-mini-2025-04-16', object='chat.completion', service_tier='d

Trace(trace_id=tr-d204ecec886da1f03823c5c5dce9686c)

In [6]:
# Load the prompt
prompt = mlflow.genai.load_prompt("prompts:/test_prompt/1")

'Translate everything after the colon into Spanish : Hello, how are you?'

In [ ]:


# Use the prompt with an LLM
client = OpenAI()
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt.format(english_text="Hello, how are you?"),
        }
    ],
    model="gpt-4o-mini",
)

print(response.choices[0].message.content)

Hola, ¿cómo estás?


Trace(trace_id=tr-f112a73b697d971e3b799f14dca052d0)

## Do some evals [link](https://mlflow.org/docs/latest/genai/eval-monitor/quickstart/#prerequisites)

In [ ]:
def qa_predict_fn(question: str) -> str:
    # The name queation here must match the key in the eval dataset
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer questions concisely.",
            },
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


In [13]:
qa_predict_fn('What is the capital of France?')

'The capital of France is Paris.'

Trace(trace_id=tr-430528d80a4dbb2cbbf0efe5834dcc67)

In [23]:
eval_dataset = [
    # Question ahs to match the dummy variable name in the prediction function
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"},
    },
    {
        "inputs": {"question": "Who was the first person to build an airplane?"},
        "expectations": {"expected_response": "Wright Brothers"},
    },
    {
        "inputs": {"question": "Who wrote Romeo and Juliet?"},
        "expectations": {"expected_response": "William Shakespeare"},
    },
]

In [20]:
@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs.split()) <= 10

scorers = [is_concise]

In [21]:
results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=qa_predict_fn,
        scorers=scorers,
    )

2026/03/12 15:37:18 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating: 100%|██████████| 3/3 [Elapsed: 00:02, Remaining: 00:00] 


In [22]:
results

EvaluationResult(
  run_id: 1427e036a2524085a39a3e63f3505b14
  metrics:
    is_concise/mean: 0.6666666666666666
  result_df: 3 rows x 14 cols
)